# Predicting State-wise EV Adoption  
## Notebook 03: Feature Engineering

### Purpose
This notebook focuses on the creation of new features that would further enhance the information provided by the dataset in order to conduct EDA.

The goal is to make new features using the existing available data from the cleaned dataset to further draw out the hidden data from the the dataset in EDA.

In [1]:
# get a view of the cleaned dataset
import pandas as pd
import numpy as np
df = pd.read_csv("../Data/ev_adoption_cleaned.csv")
df.head()

,country,region,year,vehicle_segment,ev_sales(units),petrol_car_sales(units),diesel_car_sales(units),total_vehicle_sales(units),ev_market_share(%),charging_stations(units),...,average_ev_range(km),fuel_price(usd/ltr),electricity_price(usd/kwh),gdp(usd/person),urban_population(%),co2_emissions(mt),ev_subsidy(usd),emission_regulation(scale_0-100),ev_growth_rate_yoy(%),ev_domination(0/1)
0,Australia,Oceania,2010,commercial,5,92877,61921,154803,0.00,0,...,124,1.09,0.149,51977,88.8,88.7,0,30.4,0.00,0
1,Australia,Oceania,2010,mass_market,57,535933,73089,609079,0.01,0,...,124,1.09,0.149,51977,88.8,88.7,0,30.4,0.00,0
2,Australia,Oceania,2010,premium,37,235282,20462,255781,0.01,0,...,124,1.09,0.149,51977,88.8,88.7,0,30.4,0.00,0
3,Australia,Oceania,2011,commercial,11,98092,65395,163498,0.01,105,...,133,1.09,0.163,52807,88.9,88.3,0,30.8,120.00,0
4,Australia,Oceania,2011,mass_market,129,569679,77684,647492,0.02,105,...,133,1.09,0.163,52807,88.9,88.3,0,30.8,126.32,0


In [2]:
# checking dataset structure issues
df.shape

(1200, 21)

In [3]:
# adds new feature representing ice sales
df.insert(
    loc=df.columns.get_loc("diesel_car_sales(units)") + 1,
    column="ice_total_sales(units)",
    value=(
        df["petrol_car_sales(units)"] +
        df["diesel_car_sales(units)"]
    )
)

In [4]:
# adds a new feature comparing ev and ice sales in ratio(>1 means ev sales higher, <1 means ice sales higher)
df.insert(
    loc=df.columns.get_loc("total_vehicle_sales(units)") + 1,
    column="ev_ice_ratio",
    value=(
         df["ev_sales(units)"] /
         df["ice_total_sales(units)"]
    )
)

In [5]:
# adds a new feature that measures the support availability for ev vehicles(>1 means ev has more charging points, <1 means ev has less chargning points)
df.insert(
    loc=df.columns.get_loc("fast_chargers_share(%)") + 1,
    column="chargers_per_ev_ratio",
    value=(
        df["charging_stations(units)"] /
        df["ev_sales(units)"]
    )
)

In [6]:
# adds a new feature using binning to categorise ev growth
df.insert(
    loc=df.columns.get_loc("ev_growth_rate_yoy(%)") + 1,
    column="ev_growth_rate_category",
    value=pd.cut(
        df["ev_growth_rate_yoy(%)"],
        bins=[-np.inf, 0, 20, 50, 100, np.inf],
        labels=[
            "Decline",
            "Slow Growth",
            "Moderate Growth",
            "Fast Growth",
            "Extraordinary Growth"
        ]
    )
)

In [7]:
# adds a new feature comparing fuel and electricity price(>1 means fuel price higher, <1 means electricity price higher)
df.insert(
    loc=df.columns.get_loc("electricity_price(usd/kwh)") + 1,
    column="fuel_to_electric_ratio",
    value=(
         df["fuel_price(usd/ltr)"] /
         df["electricity_price(usd/kwh)"]
    )
)

In [8]:
# adds a new feature that evaluates the urban population strength to afford ev vehicles
df["economic_index"] = (
    (df["gdp(usd/person)"] *
    df["urban_population(%)"]) / 1000
)

In [9]:
# adds a new feature that evaluates the strenght of the policies in support of ev vehicles
df["policy_index"] = (
    (df["ev_subsidy(usd)"] *
    df["emission_regulation(scale_0-100)"]) / 100
)

In [10]:
# adds a new feature that shifts the policy index by 1 year to capture the delayed effect of policies on EV adoption
df["policy_index_lagged_1y"] = (
    df.sort_values(by=["country", "vehicle_segment", "year"])
    .groupby(["country", "vehicle_segment"])["policy_index"]
    .shift(1)
)

In [11]:
# adds a new feature that calculates a ratio expressing the country's strictness towards pollution relative to its emissions
df["environmental_stringency_ratio"] = (
    df["co2_emissions(mt)"] / (df["emission_regulation(scale_0-100)"] + 1) 
)


In [12]:
# adds a new feature that represents years from smallest to largest using 0,1,2,etc. respectfully(for modelling efficiency)
df.insert(
    loc=df.columns.get_loc("year") + 1,
    column="year_normalized",
    value=(
        df["year"] - df["year"].min()
    )
)

In [13]:
# adds a new feature that evaluates the change in value of ev market share between each year for the same country and segment
df.insert(
    loc=df.columns.get_loc("ev_market_share(%)") + 1,
    column="market_share_change(%)",
    value=(
        df.sort_values(by=["country", "vehicle_segment", "year"])
        .groupby(["country", "vehicle_segment"])["ev_market_share(%)"]
        .diff()
    )
)


In [14]:
# checking the new features
df.columns

Index(['country', 'region', 'year', 'year_normalized', 'vehicle_segment',
       'ev_sales(units)', 'petrol_car_sales(units)', 'diesel_car_sales(units)',
       'ice_total_sales(units)', 'total_vehicle_sales(units)', 'ev_ice_ratio',
       'ev_market_share(%)', 'market_share_change(%)',
       'charging_stations(units)', 'fast_chargers_share(%)',
       'chargers_per_ev_ratio', 'average_ev_range(km)', 'fuel_price(usd/ltr)',
       'electricity_price(usd/kwh)', 'fuel_to_electric_ratio',
       'gdp(usd/person)', 'urban_population(%)', 'co2_emissions(mt)',
       'ev_subsidy(usd)', 'emission_regulation(scale_0-100)',
       'ev_growth_rate_yoy(%)', 'ev_growth_rate_category',
       'ev_domination(0/1)', 'economic_index', 'policy_index',
       'policy_index_lagged_1y', 'environmental_stringency_ratio'],
      dtype='object')

In [15]:
# replacing infinity values generated by ratio features with NaN, then filling all missing values with 0
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(0, inplace=True)

In [16]:
# checking for missing values
df.isna().sum()

country                             0
region                              0
year                                0
year_normalized                     0
vehicle_segment                     0
ev_sales(units)                     0
petrol_car_sales(units)             0
diesel_car_sales(units)             0
ice_total_sales(units)              0
total_vehicle_sales(units)          0
ev_ice_ratio                        0
ev_market_share(%)                  0
market_share_change(%)              0
charging_stations(units)            0
fast_chargers_share(%)              0
chargers_per_ev_ratio               0
average_ev_range(km)                0
fuel_price(usd/ltr)                 0
electricity_price(usd/kwh)          0
fuel_to_electric_ratio              0
gdp(usd/person)                     0
urban_population(%)                 0
co2_emissions(mt)                   0
ev_subsidy(usd)                     0
emission_regulation(scale_0-100)    0
ev_growth_rate_yoy(%)               0
ev_growth_ra

In [17]:
# checking for data type consistency and structure
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 32 columns):
 #   Column                            Non-Null Count  Dtype   
---  ------                            --------------  -----   
 0   country                           1200 non-null   object  
 1   region                            1200 non-null   object  
 2   year                              1200 non-null   int64   
 3   year_normalized                   1200 non-null   int64   
 4   vehicle_segment                   1200 non-null   object  
 5   ev_sales(units)                   1200 non-null   int64   
 6   petrol_car_sales(units)           1200 non-null   int64   
 7   diesel_car_sales(units)           1200 non-null   int64   
 8   ice_total_sales(units)            1200 non-null   int64   
 9   total_vehicle_sales(units)        1200 non-null   int64   
 10  ev_ice_ratio                      1200 non-null   float64 
 11  ev_market_share(%)                1200 non-null   float6

,country,region,year,year_normalized,vehicle_segment,ev_sales(units),petrol_car_sales(units),diesel_car_sales(units),ice_total_sales(units),total_vehicle_sales(units),...,co2_emissions(mt),ev_subsidy(usd),emission_regulation(scale_0-100),ev_growth_rate_yoy(%),ev_growth_rate_category,ev_domination(0/1),economic_index,policy_index,policy_index_lagged_1y,environmental_stringency_ratio
0,Australia,Oceania,2010,0,commercial,5,92877,61921,154798,154803,...,88.7,0,30.4,0.00,Decline,0,4615.5576,0.0,0.0,2.824841
1,Australia,Oceania,2010,0,mass_market,57,535933,73089,609022,609079,...,88.7,0,30.4,0.00,Decline,0,4615.5576,0.0,0.0,2.824841
2,Australia,Oceania,2010,0,premium,37,235282,20462,255744,255781,...,88.7,0,30.4,0.00,Decline,0,4615.5576,0.0,0.0,2.824841
3,Australia,Oceania,2011,1,commercial,11,98092,65395,163487,163498,...,88.3,0,30.8,120.00,Extraordinary Growth,0,4694.5423,0.0,0.0,2.776730
4,Australia,Oceania,2011,1,mass_market,129,569679,77684,647363,647492,...,88.3,0,30.8,126.32,Extraordinary Growth,0,4694.5423,0.0,0.0,2.776730


In [18]:
# save feature added dataset
df.to_csv("../Data/ev_adoption_featured.csv", index=False)

### Observations Summary

- Added 11 new features:
1. `ice_total_sales(units)`
2. `ev_ice_ratio`
3. `market_share_change(%)`
4. `chargers_per_ev_ratio`
5. `fuel_to_electric_ratio`
6. `ev_growth_rate_category`
7. `economic_index`
8. `policy_index`
9. `year_normalized`
10. `policy_index_lagged_1y`
11. `environmental_stringency_ratio`
- In the prediction stage, the following features will likely be removed due to data leakage and multicollinearity issues:
1. `ev_sales(units)`
2. `ev_market_share(%)`
3. `market_share_change(%)`
4. `ev_growth_rate_yoy(%)`
5. `ev_growth_rate_category`
6. `total_vehicle_sales(units)`
7. `ev_ice_ratio`
8. `chargers_per_ev_ratio`
9. `petrol_car_sales(units)`
10. `diesel_car_sales(units)`
11. `gdp(usd/person)`
12. `urban_population(%)`
13. `ev_subsidy(usd)`
14. `emission_regulation(scale_0-100)`
15. `year`

### Next Step

Proceed to Notebook 04: EDA